<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/Graph_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ts2vg
!pip install mne
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from ts2vg import HorizontalVG
import scipy.sparse as sp

In [ ]:
def build_multiplex_hvg_edges(window_df):
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes   = len(electrode_cols)
    n_timepoints   = len(window_df)
    n_nodes_total  = n_electrodes * n_timepoints

    print(f"Electrodes   : {n_electrodes}")
    print(f"Time points  : {n_timepoints}")
    print(f"Total nodes  : {n_nodes_total}")

    # 1. NODE INDEX
    def node_id(layer, t):
        return layer * n_timepoints + t

    node_index = {
        (electrode_cols[l], t): node_id(l, t)
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    }

    # 2. INTRA-LAYER EDGES
    intra_edges = []
    layer_info  = {}

    for l, electrode in enumerate(electrode_cols):
        ts  = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
            u = node_id(l, t_i)
            v = node_id(l, t_j)
            intra_edges.append({
                "source"       : u,
                "target"       : v,
                "source_label" : f"{electrode}_t{t_i}",
                "target_label" : f"{electrode}_t{t_j}",
                "edge_type"    : "intra",
                "layer"        : electrode,
            })

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index"  : l,
        }

    print(f"Intra-layer edges : {len(intra_edges)}")

    # 3. INTER-LAYER EDGES
    inter_edges = []

    for t in range(n_timepoints):
        for l_i in range(n_electrodes):
            for l_j in range(l_i + 1, n_electrodes):
                u = node_id(l_i, t)
                v = node_id(l_j, t)
                inter_edges.append({
                    "source"       : u,
                    "target"       : v,
                    "source_label" : f"{electrode_cols[l_i]}_t{t}",
                    "target_label" : f"{electrode_cols[l_j]}_t{t}",
                    "edge_type"    : "inter",
                    "layer"        : f"{electrode_cols[l_i]}<->{electrode_cols[l_j]}",
                })

    print(f"Inter-layer edges : {len(inter_edges)}")

    # 4. COMBINED EDGE LIST
    edge_list = pd.DataFrame(intra_edges + inter_edges)
    print(f"Total edges       : {len(edge_list)}")

    node_labels = [
        f"{electrode_cols[l]}_t{t}"
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    ]

    graph_meta = {
        "electrode_cols": electrode_cols,
        "n_electrodes": n_electrodes,
        "n_timepoints": n_timepoints,
        "n_nodes_total": n_nodes_total,
        "time_index": window_df.index.to_numpy(),
    }

    return edge_list, node_index, layer_info, node_labels, graph_meta